# TARGETED WIENER RERUN

**File**: `05_visa_patchcore_clean_wiener.ipynb`

This notebook is a surgically stripped-down version of the original pipeline.
It is designed solely to regenerate the misspecified Wiener deconvolution rows
using the corrected, per-severity PSF parameters.

**It ONLY executes:**
- `gaussian_blur` and `motion_blur`
- `mild` and `moderate` severities
- Wiener deconvolution rescue

All other corruptions, severities, and rescue methods have been removed for speed.

In [ ]:
import os
import sys

# ---------------------------------------------------------------------------
# Fix: Disable rich/tqdm progress bars that cause RecursionError on Kaggle.
# ---------------------------------------------------------------------------
os.environ["ANOMALIB_USE_RICH"] = "0"
os.environ["RICH_NO_THEME"] = "1"
sys.setrecursionlimit(5000)

import time
import json
import gc
import numpy as np
import pandas as pd
import cv2
import albumentations as A
import torch
from torch.utils.data import Dataset


In [ ]:
# ---------------------------------------------------------------------------
# 0. Global Setup & Timeout Logic
# ---------------------------------------------------------------------------
START_TIME = time.time()
TIMEOUT_SECONDS = 11.5 * 3600

import os
import shutil
import time

# Target the exact path throwing the error
ORIGINAL_DATA_PATH = "/kaggle/input/datasets/ess1004/visa-anomaly-detection"

# Fallback just in case Kaggle mounted it at the standard root
if not os.path.exists(ORIGINAL_DATA_PATH):
    ORIGINAL_DATA_PATH = "/kaggle/input/visa-anomaly-detection"

# Create a brand new writable path to avoid any cached folder confusion
VISA_ROOT = "/kaggle/working/visa_writable_copy"

# Copy dataset to working directory
if not os.path.exists(VISA_ROOT):
    print(f"Copying dataset from {ORIGINAL_DATA_PATH} to {VISA_ROOT}...")
    shutil.copytree(ORIGINAL_DATA_PATH, VISA_ROOT, dirs_exist_ok=True)
    print("Dataset successfully copied to writable directory!")

CONFIG_PATH = "/kaggle/input/notebooks/hasanmahmudabdullah/03-severity-calibration/experiment_config.json"
OUTPUT_FILE = "/kaggle/working/results/visa_patchcore.csv"
PARTIAL_FILE = "/kaggle/working/results/visa_patchcore_partial.csv"

CATEGORIES = [
    "candle", "capsules", "cashew", "chewinggum",
    "fryum", "macaroni1", "macaroni2",
    "pcb1", "pcb2", "pcb3", "pcb4", "pipe_fryum",
]
SEEDS = [42, 123, 456]

print(f"Script started at {time.ctime(START_TIME)}")
print(f"Graceful timeout set to {TIMEOUT_SECONDS / 3600:.1f} hours.")
print(f"Using Writable Dataset Root: {VISA_ROOT}")

def check_timeout():
    elapsed = time.time() - START_TIME
    if elapsed > TIMEOUT_SECONDS:
        print(f"\n{'!'*60}\nTIMEOUT REACHED ({elapsed/3600:.1f}h). Exiting gracefully.\n{'!'*60}")
        save_results()
        sys.exit(0)

def save_results():
    if 'all_results' in globals() and all_results:
        os.makedirs("/kaggle/working/results", exist_ok=True)
        df = pd.DataFrame(all_results)
        df.to_csv(OUTPUT_FILE, index=False)
        df.to_csv(PARTIAL_FILE, index=False)

def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
\n

In [ ]:
# ---------------------------------------------------------------------------
# 1. Dependency Check — same pattern as run_patchcore.py
# ---------------------------------------------------------------------------
def ensure_dependencies():
    import subprocess
    packages = ["anomalib", "lightning", "albumentationsx", "scikit-image", "opencv-python-headless"]
    for package in packages:
        try:
            check_name = "cv2" if package == "opencv-python-headless" else (
                package.replace("-", "_") if package != "albumentationsx" else "albumentations")
            __import__(check_name)
        except ImportError:
            print(f"Installing missing dependency: {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

ensure_dependencies()

# Now safe to import
import lightning as L
from anomalib.engine import Engine
from anomalib.models import Patchcore
try:
    from anomalib.data import Visa as VisaDataModule
    HAS_VISA_DM = True
    print("Using Anomalib native Visa datamodule.")
except ImportError:
    from anomalib.data import MVTecAD as VisaDataModule
    HAS_VISA_DM = False
    print("Anomalib Visa DM not found — falling back to MVTecAD loader (requires VisA in MVTec layout).")


In [ ]:
# ---------------------------------------------------------------------------
# 2. Embedded Corruption & Rescue Functions (Stripped down for Wiener rerun)
# ---------------------------------------------------------------------------

def apply_gaussian_blur(image, sigma, kernel_size):
    t = A.GaussianBlur(blur_limit=(kernel_size, kernel_size), sigma_limit=(sigma, sigma), p=1.0)
    return t(image=image)["image"]

def apply_motion_blur(image, kernel_size):
    t = A.MotionBlur(blur_limit=(kernel_size, kernel_size), p=1.0)
    return t(image=image)["image"]

def apply_corruption(image, ctype, severity, config, seed=42):
    params = config["corruptions"][ctype][severity]
    if ctype == "gaussian_blur": return apply_gaussian_blur(image, params["sigma"], params["kernel_size"])
    elif ctype == "motion_blur": return apply_motion_blur(image, params["kernel_size"])
    else: raise ValueError(f"Unknown corruption type: {ctype}")

def apply_wiener_deconv(image: np.ndarray, sigma: float, kernel_size: int, balance: float = 0.1) -> np.ndarray:
    import cv2, numpy as np
    from skimage.restoration import wiener
    psf = np.zeros((kernel_size, kernel_size))
    center = kernel_size // 2
    psf[center, center] = 1.0
    psf = cv2.GaussianBlur(psf, (kernel_size, kernel_size), sigmaX=sigma, sigmaY=sigma)
    psf /= psf.sum()
    out = np.zeros_like(image, dtype=np.float64)
    for i in range(3):
        out[:, :, i] = wiener(image[:, :, i] / 255.0, psf, balance, clip=False)
    out = np.clip(out * 255, 0, 255).astype(np.uint8)
    return out

def apply_motion_wiener_deconv(image: np.ndarray, kernel_size: int, balance: float = 0.1) -> np.ndarray:
    import cv2, numpy as np
    from skimage.restoration import wiener
    psf = np.zeros((kernel_size, kernel_size))
    psf[kernel_size // 2, :] = 1.0 / kernel_size
    angle = 0
    M = cv2.getRotationMatrix2D((kernel_size / 2, kernel_size / 2), angle, 1)
    psf = cv2.warpAffine(psf, M, (kernel_size, kernel_size))
    psf /= psf.sum()
    out = np.zeros_like(image, dtype=np.float64)
    for i in range(3):
        out[:, :, i] = wiener(image[:, :, i] / 255.0, psf, balance, clip=False)
    out = np.clip(out * 255, 0, 255).astype(np.uint8)
    return out

def get_rescue_map(severity, config):
    gauss_p = config["corruptions"]["gaussian_blur"][severity]
    motion_p = config["corruptions"]["motion_blur"][severity]
    return {
        "gaussian_blur": [("Wiener", lambda img, s=gauss_p["sigma"], k=gauss_p["kernel_size"]:
                           apply_wiener_deconv(img, sigma=s, kernel_size=k))],
        "motion_blur": [("Wiener (Motion PSF)", lambda img, k=motion_p["kernel_size"]:
                         apply_motion_wiener_deconv(img, kernel_size=k))]
    }


In [ ]:
# ---------------------------------------------------------------------------
# 3. Load Configuration
# ---------------------------------------------------------------------------
for cfg_path in [CONFIG_PATH, "experiment_config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path) as f:
            config = json.load(f)
        print(f"Config loaded from: {cfg_path}")
        break
else:
    raise FileNotFoundError(f"experiment_config.json not found. Add severity calibration output as Kaggle input.")


In [ ]:
# ---------------------------------------------------------------------------
# 4. Corrupted Dataset Wrapper — identical to run_patchcore.py
# ---------------------------------------------------------------------------
class CorruptedDatasetWrapper(Dataset):
    def __init__(self, base_dataset, ctype, severity, config, rescue_func=None):
        self.base_dataset = base_dataset
        self.ctype = ctype
        self.severity = severity
        self.config = config
        self.rescue_func = rescue_func

    def __len__(self): return len(self.base_dataset)

    def __getattr__(self, name):
        return getattr(self.base_dataset, name)

    def __getitem__(self, idx):
        import dataclasses
        item = self.base_dataset[idx]
        if dataclasses.is_dataclass(item): image = item.image
        else: image = item["image"]

        if isinstance(image, torch.Tensor):
            img_np = image.permute(1, 2, 0).cpu().numpy()
            if img_np.max() <= 1.0: img_np = (img_np * 255).astype(np.uint8)
            else: img_np = img_np.astype(np.uint8)
        else:
            img_np = np.array(image).astype(np.uint8)

        corrupted = apply_corruption(img_np, self.ctype, self.severity, self.config, seed=42 + idx)
        final_img = self.rescue_func(corrupted) if self.rescue_func else corrupted
        final_tensor = torch.from_numpy(final_img).permute(2, 0, 1).float() / 255.0

        if dataclasses.is_dataclass(item): return dataclasses.replace(item, image=final_tensor)
        else: item["image"] = final_tensor; return item


In [ ]:
# ---------------------------------------------------------------------------
# 5. Engine & Resume Logic — identical to run_patchcore.py
# ---------------------------------------------------------------------------
class DisableCheckpointing(L.Callback):
    def setup(self, trainer, pl_module, stage):
        from lightning.pytorch.callbacks import ModelCheckpoint
        trainer.callbacks = [cb for cb in trainer.callbacks if not isinstance(cb, ModelCheckpoint)]

def make_engine():
    return Engine(max_epochs=1, accelerator="auto", devices=1,
                  default_root_dir="/tmp/anomalib", enable_progress_bar=False,
                  callbacks=[DisableCheckpointing()])

def safe_auroc(result_dict):
    for key in ["image_AUROC", "image_auroc", "auroc", "AUROC", "test_image_AUROC"]:
        if key in result_dict: return result_dict[key]
    print(f"  ⚠️  AUROC key not found. Keys: {list(result_dict.keys())}")
    return None

completed_keys = set()
all_results = []

os.makedirs("results", exist_ok=True)

import glob
import shutil
# Auto-discover previous partials from Kaggle input datasets
input_partials = glob.glob("/kaggle/input/**/*partial*.csv", recursive=True)
if input_partials:
    latest_partial = max(input_partials, key=os.path.getmtime)
    os.makedirs(os.path.dirname(PARTIAL_FILE), exist_ok=True)
    shutil.copy(latest_partial, PARTIAL_FILE)
    print(f"Auto-restored partial progress from: {latest_partial}")

for p in [OUTPUT_FILE, PARTIAL_FILE]:
    if os.path.exists(p):
        try:
            old_df = pd.read_csv(p)
            
            # Group by category and seed to count rows
            counts = old_df.groupby(['category', 'seed']).size()
            valid_pairs = []
            
            for (cat, seed), count in counts.items():
                if count == 34: # Only mark as complete if all 34 rows exist
                    completed_keys.add((cat, int(seed)))
                    valid_pairs.append((cat, int(seed)))
                else:
                    print(f"Discarding partial pair {cat} (Seed {seed}) with {count}/34 rows to re-run it cleanly.")
            
            # Keep only the rows for fully completed pairs to prevent duplicates
            clean_df = old_df[old_df.apply(lambda row: (row['category'], int(row['seed'])) in valid_pairs, axis=1)]
            all_results = clean_df.to_dict("records")
            
            print(f"Resumed {len(completed_keys)} fully completed pairs from {p}.")
            break
        except Exception as e:
            print(f"Could not load existing results: {e}")


In [ ]:
# ---------------------------------------------------------------------------
# 6. Main Loop
# ---------------------------------------------------------------------------
print(f"\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VisA root: {VISA_ROOT}")
print(f"Categories: {CATEGORIES}")

for category in CATEGORIES:
    for seed in SEEDS:
        check_timeout()
        if (category, int(seed)) in completed_keys:
            print(f"⏩ Skipping {category} (Seed {seed})")
            continue

        print(f"\n{'='*60}\nCATEGORY: {category.upper()} | SEED: {seed}\n{'='*60}")
        L.seed_everything(seed)

        engine = make_engine()
        model = Patchcore(backbone="wide_resnet50_2", num_neighbors=9)
        datamodule = VisaDataModule(root=VISA_ROOT, category=category,
                                    train_batch_size=32, eval_batch_size=32)
        try:
            print(f"[PHASE 1] Training Baseline...")
            engine.fit(model=model, datamodule=datamodule)
            res = engine.test(model=model, datamodule=datamodule)
            clean_auroc = safe_auroc(res[0])
            # all_results.append({"model": "PatchCore", "dataset": "VisA", "category": category,
                                 "seed": seed, "phase": "baseline", "ctype": "clean",
                                 "severity": "none", "rescue": "none", "image_AUROC": clean_auroc})
            save_results()

            for ctype, severities in config["corruptions"].items():
                for sev in severities.keys():
                    print(f"[PHASE 2] Degradation: {ctype} ({sev})")
                    dm_base = VisaDataModule(root=VISA_ROOT, category=category,
                                             train_batch_size=32, eval_batch_size=32)
                    dm_base.setup(stage="test")
                    from torch.utils.data import DataLoader
                    deg_loader = DataLoader(
                        CorruptedDatasetWrapper(dm_base.test_data, ctype, sev, config),
                        batch_size=32, num_workers=2, collate_fn=dm_base.test_data.collate_fn)
                    res = engine.test(model=model, dataloaders=deg_loader)
                    # all_results.append({"model": "PatchCore", "dataset": "VisA", "category": category,
                                        "seed": seed, "phase": "degradation", "ctype": ctype,
                                        "severity": sev, "rescue": "none", "image_AUROC": safe_auroc(res[0])})
                    save_results()

                    if ctype in get_rescue_map(sev, config):
                        for r_name, r_func in get_rescue_map(sev, config)[ctype]:
                            print(f"[PHASE 3] Rescue: {ctype} ({sev}) + {r_name}")
                            dm_r = VisaDataModule(root=VISA_ROOT, category=category,
                                                  train_batch_size=32, eval_batch_size=32)
                            dm_r.setup(stage="test")
                            res_loader = DataLoader(
                                CorruptedDatasetWrapper(dm_r.test_data, ctype, sev, config, rescue_func=r_func),
                                batch_size=32, num_workers=2, collate_fn=dm_r.test_data.collate_fn)
                            res = engine.test(model=model, dataloaders=res_loader)
                            # all_results.append({"model": "PatchCore", "dataset": "VisA", "category": category,
                                                "seed": seed, "phase": "rescue", "ctype": ctype,
                                                "severity": sev, "rescue": r_name,
                                                "image_AUROC": safe_auroc(res[0])})
                            save_results()

        except Exception as e:
            print(f"❌ Error in {category}/{seed}: {e}")

        del model
        del engine
        cleanup_memory()

save_results()
print(f"\n{'='*60}\nVISA PATCHCORE COMPLETE — {len(all_results)} rows saved.\n{'='*60}")
\n